# Long-horizon anchored-L2 on the frozen 24-condition benchmark

Matched protocol:
- 6 datasets × H={96,192,336,720}
- exact Exp33 frozen query/channel manifests
- same-channel memory
- train-only normalization
- L=96, memory stride=8, Top-K=10
- candidate future fully observed before the query memory boundary
- anchored L2 searches the full admissible same-channel memory

Anchored shape: \(\tilde x_t=x_t-x_L\). Candidates are ranked by squared
Euclidean distance between anchored past windows.

The notebook caches each condition and finally compares Anchored L2 with the
current stride-8 Pearson and full PRR frozen results.

In [1]:
from pathlib import Path
import io, time, gc
import numpy as np
import pandas as pd

DATASETS=("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS=(96,192,336,720)
L=96
MEMORY_STRIDE=8
TOP_K=10

manifest_candidates=[
    Path("/data/code/which_histories_matter_ext/results/exp33/final_eval_manifest"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster/results/exp33/final_eval_manifest"),
]
for base in [Path("/data/code"),Path("/data")]:
    if base.exists():
        manifest_candidates += list(base.glob("*/results/exp33/final_eval_manifest"))
        manifest_candidates += list(base.glob("*/*/results/exp33/final_eval_manifest"))

MANIFEST_ROOT=next((p for p in manifest_candidates if p.exists()),None)
assert MANIFEST_ROOT is not None, "Exp33 frozen manifests not found"

PROJECT_ROOT=MANIFEST_ROOT.parents[2]
OUT_ROOT=PROJECT_ROOT/"results"/"anchored_l2_long_horizon_frozen24"
CACHE_ROOT=OUT_ROOT/"cache"
OUT_ROOT.mkdir(parents=True,exist_ok=True)
CACHE_ROOT.mkdir(parents=True,exist_ok=True)

print("MANIFEST_ROOT:",MANIFEST_ROOT)
print("OUT_ROOT:",OUT_ROOT)

MANIFEST_ROOT: /data/code/which_histories_matter_ext/results/exp33/final_eval_manifest
OUT_ROOT: /data/code/which_histories_matter_ext/results/anchored_l2_long_horizon_frozen24


In [2]:
EXPECTED={"ETTh1":7,"Weather":21,"Electricity":321,"Traffic":862,"Exchange":8,"Solar":137}

CANDIDATES={
  "ETTh1":[
    "/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv",
    "/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv",
    "/data/dataset/ETT-small/ETTh1.csv","/data/dataset/ETTh1.csv"],
  "Weather":[
    "/data/Time-Series-Library/dataset/weather/weather.csv",
    "/data/Time-Series-Library_v2/dataset/weather/weather.csv",
    "/data/dataset/weather/weather.csv","/data/dataset/weather.csv"],
  "Electricity":[
    "/data/Time-Series-Library/dataset/electricity/electricity.csv",
    "/data/Time-Series-Library_v2/dataset/electricity/electricity.csv",
    "/data/dataset/electricity/electricity.csv","/data/dataset/electricity.csv"],
  "Traffic":[
    "/data/Time-Series-Library/dataset/traffic/traffic.csv",
    "/data/Time-Series-Library_v2/dataset/traffic/traffic.csv",
    "/data/dataset/traffic/traffic.csv","/data/dataset/traffic.csv"],
  "Exchange":[
    "/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv",
    "/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv",
    "/data/dataset/exchange_rate/exchange_rate.csv","/data/dataset/exchange_rate.csv"],
  "Solar":[
    "/data/Time-Series-Library/dataset/Solar/solar_AL.txt",
    "/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt",
    "/data/dataset/Solar/solar_AL.txt","/data/dataset/solar_AL.txt"],
}

def resolve(name):
    for s in CANDIDATES[name]:
        p=Path(s)
        if p.is_file(): return p
    raise FileNotFoundError(name)

def clean(df):
    return (df.apply(pd.to_numeric,errors="coerce")
              .dropna(axis=1,how="all")
              .replace([np.inf,-np.inf],np.nan)
              .interpolate(axis=0,limit_direction="both")
              .ffill().bfill())

def load_data(name):
    p=resolve(name)
    if name=="Solar":
        df=pd.read_csv(p,header=None)
        if df.shape[1]==1:
            df=pd.read_csv(p,header=None,sep=r"\s+")
        num=clean(df)
    else:
        df=pd.read_csv(p)
        date=next((c for c in df.columns if str(c).lower() in {"date","datetime","timestamp","time"}),None)
        val=df.drop(columns=[date]) if date is not None else df.copy()
        if name in {"Weather","Electricity"} and "OT" in val.columns:
            val=val[[c for c in val.columns if c!="OT"]+["OT"]]
        num=clean(val)

    raw=num.to_numpy(np.float32)
    assert raw.shape[1]==EXPECTED[name],(name,raw.shape)
    n=len(raw)
    if name=="ETTh1":
        assert raw.shape==(17420,7)
        train_end,val_end=8640,11520
    else:
        train_end=int(0.70*n)
        val_end=n-int(0.20*n)
    mu=raw[:train_end].mean(0).astype(np.float32)
    sd=raw[:train_end].std(0,ddof=0).astype(np.float32)
    assert np.all(sd>1e-6)
    z=((raw-mu)/sd).astype(np.float32)
    return dict(path=p,z=z,n=n,channels=raw.shape[1],train_end=train_end,val_end=val_end)

DATA={d:load_data(d) for d in DATASETS}
display(pd.DataFrame([dict(dataset=d,rows=DATA[d]["n"],channels=DATA[d]["channels"],
                           train_end=DATA[d]["train_end"],val_end=DATA[d]["val_end"],
                           path=str(DATA[d]["path"])) for d in DATASETS]))

,dataset,rows,channels,train_end,val_end,path
0,ETTh1,17420,7,8640,11520,/data/Time-Series-Library/dataset/ETT-small/ET...
1,Weather,52696,21,36887,42157,/data/Time-Series-Library/dataset/weather/weat...
2,Electricity,26304,321,18412,21044,/data/Time-Series-Library/dataset/electricity/...
3,Traffic,17544,862,12280,14036,/data/Time-Series-Library/dataset/traffic/traf...
4,Exchange,7588,8,5311,6071,/data/Time-Series-Library/dataset/exchange_rat...
5,Solar,52560,137,36792,42048,/data/Time-Series-Library/dataset/Solar/solar_...


In [3]:
def load_manifest(dataset,horizon,split):
    p=MANIFEST_ROOT/f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(),p
    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64)
        c=np.asarray(z["channel"],np.int64)
    assert a.shape==c.shape and len(a)>0
    return a,c

audit=[]
for d in DATASETS:
    for h in HORIZONS:
        for split in ("val","test"):
            a,c=load_manifest(d,h,split)
            audit.append(dict(dataset=d,horizon=h,split=split,pairs=len(a),channels=len(np.unique(c))))
display(pd.DataFrame(audit))

,dataset,horizon,split,pairs,channels
0,ETTh1,96,val,4096,7
1,ETTh1,96,test,8192,7
2,ETTh1,192,val,4096,7
3,ETTh1,192,test,8192,7
4,ETTh1,336,val,4096,7
5,ETTh1,336,test,8192,7
6,ETTh1,720,val,4096,7
7,ETTh1,720,test,8192,7
8,Weather,96,val,4096,21
9,Weather,96,test,8192,21


In [4]:
def past_windows(z,ch,anchors):
    anchors=np.asarray(anchors,np.int64)
    idx=anchors[:,None]-L+np.arange(L)[None,:]
    return z[idx,int(ch)].astype(np.float32)

def future_residuals(z,ch,anchors,h):
    anchors=np.asarray(anchors,np.int64)
    idx=anchors[:,None]+np.arange(h)[None,:]
    fut=z[idx,int(ch)].astype(np.float32)
    cur=z[anchors-1,int(ch)].astype(np.float32)
    return (fut-cur[:,None]).astype(np.float32)

def anchored_shape(x):
    x=np.asarray(x,np.float32)
    return (x-x[...,-1:]).astype(np.float32)

def cache_path(d,h,split):
    return CACHE_ROOT/f"{d}_H{h}_{split}_anchored_l2.npz"

def evaluate_condition(dataset,horizon,split="test",query_chunk=128):
    qa,qc=load_manifest(dataset,horizon,split)
    p=cache_path(dataset,horizon,split)

    if p.exists():
        with np.load(p,allow_pickle=False) as zc:
            ca=np.asarray(zc["anchor"],np.int64)
            cc=np.asarray(zc["channel"],np.int64)
            af=np.asarray(zc["analog_mse"],np.float32)
            rf=np.asarray(zc["retrieval_forecast_mse"],np.float32)
            stride=int(np.asarray(zc["memory_stride"]).item())
        if np.array_equal(ca,qa) and np.array_equal(cc,qc) and stride==MEMORY_STRIDE:
            return dict(dataset=dataset,horizon=horizon,split=split,pairs=len(qa),
                        anchored_l2_analog_future_mse=float(af.mean()),
                        anchored_l2_retrieval_forecast_mse=float(rf.mean()),
                        seconds=0.0,cached=True)

    z=DATA[dataset]["z"]
    boundary=DATA[dataset]["train_end"] if split=="val" else DATA[dataset]["val_end"]
    af=np.full(len(qa),np.nan,np.float32)
    rf=np.full(len(qa),np.nan,np.float32)
    t0=time.perf_counter()

    for ch in np.unique(qc):
        rows=np.flatnonzero(qc==ch)
        mem_anchor=np.arange(L,boundary-horizon+1,MEMORY_STRIDE,dtype=np.int64)
        assert len(mem_anchor)>=TOP_K
        assert int(mem_anchor.max())+horizon<=boundary

        mp=past_windows(z,ch,mem_anchor)
        ms=anchored_shape(mp)
        mf=future_residuals(z,ch,mem_anchor,horizon)
        mnorm=np.sum(ms*ms,axis=1)

        for left in range(0,len(rows),query_chunk):
            rr=rows[left:left+query_chunk]
            q_anchor=qa[rr]
            qp=past_windows(z,ch,q_anchor)
            qs=anchored_shape(qp)
            qf=future_residuals(z,ch,q_anchor,horizon)

            qnorm=np.sum(qs*qs,axis=1,keepdims=True)
            d2=np.maximum(qnorm+mnorm[None,:]-2.0*(qs@ms.T),0.0)
            top=np.argpartition(d2,TOP_K-1,axis=1)[:,:TOP_K]

            chosen=mf[top]
            af[rr]=((chosen-qf[:,None,:])**2).mean(axis=2).mean(axis=1).astype(np.float32)
            mean_rel=chosen.mean(axis=1)
            rf[rr]=((mean_rel-qf)**2).mean(axis=1).astype(np.float32)

        print(f"{dataset:11s} H={horizon:3d} ch={int(ch):3d} pairs={len(rows):4d} memory={len(mem_anchor):5d}")

    assert np.isfinite(af).all() and np.isfinite(rf).all()

    np.savez_compressed(
        p,anchor=qa.astype(np.int64),channel=qc.astype(np.int16),
        analog_mse=af,retrieval_forecast_mse=rf,
        memory_boundary=np.asarray(boundary,np.int64),
        memory_stride=np.asarray(MEMORY_STRIDE,np.int64),
        top_k=np.asarray(TOP_K,np.int64),
        retrieval_name=np.asarray("last_value_anchored_L2_full_memory"),
    )

    return dict(dataset=dataset,horizon=horizon,split=split,pairs=len(qa),
                anchored_l2_analog_future_mse=float(af.mean()),
                anchored_l2_retrieval_forecast_mse=float(rf.mean()),
                seconds=float(time.perf_counter()-t0),cached=False)

In [5]:
rows=[]
for d in DATASETS:
    for h in HORIZONS:
        print("\n"+"="*90)
        print("Anchored L2",d,h)
        print("="*90)
        r=evaluate_condition(d,h,"test",128)
        rows.append(r)
        print("AFMSE =",f"{r['anchored_l2_analog_future_mse']:.6f}",
              "| RFMSE =",f"{r['anchored_l2_retrieval_forecast_mse']:.6f}")
        gc.collect()

l2_test24=pd.DataFrame(rows).sort_values(["dataset","horizon"]).reset_index(drop=True)
assert len(l2_test24)==24
display(l2_test24)
l2_test24.to_csv(OUT_ROOT/"anchored_l2_test_24conditions.csv",index=False)


Anchored L2 ETTh1 96
ETTh1       H= 96 ch=  0 pairs=1170 memory= 1417
ETTh1       H= 96 ch=  1 pairs=1170 memory= 1417
ETTh1       H= 96 ch=  2 pairs=1170 memory= 1417
ETTh1       H= 96 ch=  3 pairs=1170 memory= 1417
ETTh1       H= 96 ch=  4 pairs=1170 memory= 1417
ETTh1       H= 96 ch=  5 pairs=1171 memory= 1417
ETTh1       H= 96 ch=  6 pairs=1171 memory= 1417
AFMSE = 1.100985 | RFMSE = 0.737021

Anchored L2 ETTh1 192
ETTh1       H=192 ch=  0 pairs=1170 memory= 1405
ETTh1       H=192 ch=  1 pairs=1170 memory= 1405
ETTh1       H=192 ch=  2 pairs=1170 memory= 1405
ETTh1       H=192 ch=  3 pairs=1170 memory= 1405
ETTh1       H=192 ch=  4 pairs=1170 memory= 1405
ETTh1       H=192 ch=  5 pairs=1171 memory= 1405
ETTh1       H=192 ch=  6 pairs=1171 memory= 1405
AFMSE = 1.277228 | RFMSE = 0.828583

Anchored L2 ETTh1 336
ETTh1       H=336 ch=  0 pairs=1170 memory= 1387
ETTh1       H=336 ch=  1 pairs=1170 memory= 1387
ETTh1       H=336 ch=  2 pairs=1170 memory= 1387
ETTh1       H=336 ch=  3 pa

,dataset,horizon,split,pairs,anchored_l2_analog_future_mse,anchored_l2_retrieval_forecast_mse,seconds,cached
0,ETTh1,96,test,8192,1.100985,0.737021,0.274296,False
1,ETTh1,192,test,8192,1.277228,0.828583,0.284617,False
2,ETTh1,336,test,8192,1.446432,0.896594,0.297704,False
3,ETTh1,720,test,8192,1.714012,1.041471,0.364680,False
4,Electricity,96,test,8192,0.703011,0.536853,1.920945,False
5,Electricity,192,test,8192,0.728439,0.552923,2.471330,False
6,Electricity,336,test,8192,0.768702,0.574455,3.384150,False
7,Electricity,720,test,8192,0.815970,0.601393,5.771160,False
8,Exchange,96,test,8192,0.186254,0.104771,0.134584,False
9,Exchange,192,test,8192,0.379211,0.220980,0.152838,False


In [6]:
REFERENCE_CSV = '''dataset,horizon,pearson,prr
ETTh1,96,1.6123,1.1643
ETTh1,192,1.8175,1.2229
ETTh1,336,1.9929,1.4816
ETTh1,720,2.2541,1.4618
Weather,96,1.1613,0.4316
Weather,192,1.0628,0.5147
Weather,336,1.2910,0.6748
Weather,720,1.2958,0.8130
Electricity,96,1.1796,0.7191
Electricity,192,1.1957,0.7345
Electricity,336,1.2418,0.7648
Electricity,720,1.2877,0.8602
Traffic,96,1.9738,1.4696
Traffic,192,2.0045,1.4508
Traffic,336,2.0636,1.5147
Traffic,720,2.1665,1.7351
Exchange,96,0.2133,0.1652
Exchange,192,0.4234,0.4116
Exchange,336,0.7643,0.8094
Exchange,720,1.8868,1.7770
Solar,96,0.5887,0.5087
Solar,192,0.5979,0.5406
Solar,336,0.6238,0.5701
Solar,720,0.6385,0.5507
'''
reference=pd.read_csv(io.StringIO(REFERENCE_CSV))
reference["dataset"]=reference["dataset"].str.strip()
reference["horizon"]=reference["horizon"].astype(int)

precise=PROJECT_ROOT/"results"/"stride8_revalidation"/"table1_stride8_revalidated.csv"
if precise.exists():
    tmp=pd.read_csv(precise)
    tmp["horizon"]=tmp["horizon"].astype(int)
    need={"dataset","horizon","pearson_analog_future_mse","prr_analog_future_mse"}
    if need.issubset(tmp.columns):
        reference=tmp[["dataset","horizon","pearson_analog_future_mse","prr_analog_future_mse"]].rename(
            columns={"pearson_analog_future_mse":"pearson","prr_analog_future_mse":"prr"}
        )
        print("[REFERENCE] precise stride-8 table:",precise)
    else:
        print("[REFERENCE] embedded paper-facing values")
else:
    print("[REFERENCE] embedded paper-facing values")

display(reference)

[REFERENCE] precise stride-8 table: /data/code/which_histories_matter_ext/results/stride8_revalidation/table1_stride8_revalidated.csv


,dataset,horizon,pearson,prr
0,ETTh1,96,1.612267,1.1643
1,ETTh1,192,1.817531,1.2229
2,ETTh1,336,1.992872,1.4816
3,ETTh1,720,2.254092,1.4618
4,Electricity,96,1.179589,0.7191
5,Electricity,192,1.195698,0.7345
6,Electricity,336,1.241844,0.7648
7,Electricity,720,1.287729,0.8602
8,Exchange,96,0.213266,0.1652
9,Exchange,192,0.423408,0.4116


In [7]:
comp=l2_test24.merge(reference,on=["dataset","horizon"],how="inner")
assert len(comp)==24
comp["anchored_l2"]=comp["anchored_l2_analog_future_mse"]

comp["l2_gain_vs_pearson_pct"]=100*(comp["pearson"]-comp["anchored_l2"])/comp["pearson"]
comp["prr_gain_vs_l2_pct"]=100*(comp["anchored_l2"]-comp["prr"])/comp["anchored_l2"]
comp["l2_gain_vs_prr_pct"]=100*(comp["prr"]-comp["anchored_l2"])/comp["prr"]

def summarize(x,name):
    x=np.asarray(x,float)
    return dict(
        comparison=name,
        W=int((x>1e-12).sum()),
        T=int((np.abs(x)<=1e-12).sum()),
        L=int((x<-1e-12).sum()),
        mean_gain_pct=float(x.mean()),
        median_gain_pct=float(np.median(x)),
    )

summary=pd.DataFrame([
    summarize(comp["l2_gain_vs_pearson_pct"],"Anchored L2 vs Pearson"),
    summarize(comp["prr_gain_vs_l2_pct"],"PRR vs Anchored L2"),
    summarize(comp["l2_gain_vs_prr_pct"],"Anchored L2 vs PRR"),
])

display(comp[["dataset","horizon","pearson","anchored_l2","prr",
              "l2_gain_vs_pearson_pct","prr_gain_vs_l2_pct"]])
display(summary)

comp.to_csv(OUT_ROOT/"anchored_l2_vs_pearson_prr_24conditions.csv",index=False)
summary.to_csv(OUT_ROOT/"anchored_l2_summary.csv",index=False)

,dataset,horizon,pearson,anchored_l2,prr,l2_gain_vs_pearson_pct,prr_gain_vs_l2_pct
0,ETTh1,96,1.612267,1.100985,1.1643,31.711976,-5.750732
1,ETTh1,192,1.817531,1.277228,1.2229,29.727267,4.253613
2,ETTh1,336,1.992872,1.446432,1.4816,27.419710,-2.431354
3,ETTh1,720,2.254092,1.714012,1.4618,23.959992,14.714712
4,Electricity,96,1.179589,0.703011,0.7191,40.402065,-2.288649
5,Electricity,192,1.195698,0.728439,0.7345,39.078333,-0.831999
6,Electricity,336,1.241844,0.768702,0.7648,38.099943,0.507613
7,Electricity,720,1.287729,0.815970,0.8602,36.634978,-5.420580
8,Exchange,96,0.213266,0.186254,0.1652,12.665825,11.304072
9,Exchange,192,0.423408,0.379211,0.4116,10.438392,-8.541067


,comparison,W,T,L,mean_gain_pct,median_gain_pct
0,Anchored L2 vs Pearson,24,0,0,26.252897,27.561922
1,PRR vs Anchored L2,7,0,17,0.440643,-4.275319
2,Anchored L2 vs PRR,17,0,7,-3.656057,4.100012


In [8]:
ds=[]
for d,g in comp.groupby("dataset",sort=False):
    ds.append(dict(
        dataset=d,
        L2_vs_Pearson_W=int((g.l2_gain_vs_pearson_pct>0).sum()),
        L2_vs_Pearson_mean_pct=float(g.l2_gain_vs_pearson_pct.mean()),
        PRR_vs_L2_W=int((g.prr_gain_vs_l2_pct>0).sum()),
        PRR_vs_L2_mean_pct=float(g.prr_gain_vs_l2_pct.mean()),
    ))
ds_summary=pd.DataFrame(ds)
display(ds_summary)
ds_summary.to_csv(OUT_ROOT/"anchored_l2_dataset_summary.csv",index=False)

,dataset,L2_vs_Pearson_W,L2_vs_Pearson_mean_pct,PRR_vs_L2_W,PRR_vs_L2_mean_pct
0,ETTh1,4,28.204736,2,2.696560
1,Electricity,4,38.553830,1,-2.008404
2,Exchange,4,8.673109,2,-2.689359
3,Solar,4,15.554136,0,-4.929817
4,Traffic,4,29.431385,0,-6.388221
5,Weather,4,37.100185,2,15.963096


In [9]:
paper=comp[["dataset","horizon","pearson","anchored_l2","prr",
            "l2_gain_vs_pearson_pct","prr_gain_vs_l2_pct"]].copy()
for c in ["pearson","anchored_l2","prr"]:
    paper[c]=paper[c].map(lambda x:f"{x:.4f}")
for c in ["l2_gain_vs_pearson_pct","prr_gain_vs_l2_pct"]:
    paper[c]=paper[c].map(lambda x:f"{x:+.2f}%")

paper=paper.rename(columns={
    "dataset":"Dataset","horizon":"H","pearson":"Pearson",
    "anchored_l2":"Anchored L2","prr":"PRR (ours)",
    "l2_gain_vs_pearson_pct":"L2 gain vs. Pearson",
    "prr_gain_vs_l2_pct":"PRR gain vs. L2",
})
display(paper)
paper.to_csv(OUT_ROOT/"paper_table_anchored_l2_long_horizon.csv",index=False)

print("\nFINAL SUMMARY")
for r in summary.itertuples(index=False):
    print(f"{r.comparison}: {r.W}W/{r.T}T/{r.L}L | mean={r.mean_gain_pct:+.2f}% | median={r.median_gain_pct:+.2f}%")
print("\nSaved under:",OUT_ROOT)

,Dataset,H,Pearson,Anchored L2,PRR (ours),L2 gain vs. Pearson,PRR gain vs. L2
0,ETTh1,96,1.6123,1.1010,1.1643,+31.71%,-5.75%
1,ETTh1,192,1.8175,1.2772,1.2229,+29.73%,+4.25%
2,ETTh1,336,1.9929,1.4464,1.4816,+27.42%,-2.43%
3,ETTh1,720,2.2541,1.7140,1.4618,+23.96%,+14.71%
4,Electricity,96,1.1796,0.7030,0.7191,+40.40%,-2.29%
5,Electricity,192,1.1957,0.7284,0.7345,+39.08%,-0.83%
6,Electricity,336,1.2418,0.7687,0.7648,+38.10%,+0.51%
7,Electricity,720,1.2877,0.8160,0.8602,+36.63%,-5.42%
8,Exchange,96,0.2133,0.1863,0.1652,+12.67%,+11.30%
9,Exchange,192,0.4234,0.3792,0.4116,+10.44%,-8.54%



FINAL SUMMARY
Anchored L2 vs Pearson: 24W/0T/0L | mean=+26.25% | median=+27.56%
PRR vs Anchored L2: 7W/0T/17L | mean=+0.44% | median=-4.28%
Anchored L2 vs PRR: 17W/0T/7L | mean=-3.66% | median=+4.10%

Saved under: /data/code/which_histories_matter_ext/results/anchored_l2_long_horizon_frozen24
